In [1]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Learn the Basics](intro.html) \|\|
[Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| **Build Model** \|\|
[Autograd](autogradqs_tutorial.html) \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

学习基础知识 \|\| 快速入门 \|\| 张量 \|\| 数据集与数据加载器 \|\| 数据变换 \|\| **构建模型** \|\| 自动微分 \|\| 优化模型参数 \|\| 保存与加载模型

Build the Neural Network 构建神经网络
========================

Neural networks comprise of layers/modules that perform operations on
data. The [torch.nn](https://pytorch.org/docs/stable/nn.html) namespace
provides all the building blocks you need to build your own neural
network. Every module in PyTorch subclasses the
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
A neural network is a module itself that consists of other modules
(layers). This nested structure allows for building and managing complex
architectures easily.

神经网络由对数据执行运算的层/模块组成。[torch.nn](https://pytorch.org/docs/stable/nn.html) 命名空间提供了构建你自己的神经网络所需的全部组件。PyTorch 中的每个模块都继承自 [nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)。神经网络本身也是一个模块，它由其他模块（层）组成。这种嵌套结构让我们可以轻松构建和管理复杂的架构。

In the following sections, we\'ll build a neural network to classify
images in the FashionMNIST dataset.

在接下来的小节中，我们将构建一个神经网络，对 FashionMNIST 数据集中的图像进行分类。


In [2]:
import os
import torch
from torch import nn 
from torch.utils.data import DataLoader


Get Device for Training 获取训练设备
=======================

We want to be able to train our model on an
[accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If the current accelerator is
available, we will use it. Otherwise, we use the CPU.

我们希望能在 CUDA、MPS、MTIA 或 XPU 等[加速器](https://pytorch.org/docs/stable/torch.html#accelerators)上训练模型。如果当前有可用的加速器，就使用它；否则使用 CPU。


In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


In [4]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu" 
print(f"Using {device} device")
print(device, repr(device))

Using mps device
mps device(type='mps')


Define the Class 定义类
================

We define our neural network by subclassing `nn.Module`, and initialize
the neural network layers in `__init__`. Every `nn.Module` subclass
implements the operations on input data in the `forward` method.

我们通过继承 `nn.Module` 来定义神经网络，并在 `__init__` 中初始化神经网络的各层。每个 `nn.Module` 子类都在 `forward` 方法中实现对输入数据的运算。


In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


We create an instance of `NeuralNetwork`, and move it to the `device`,
and print its structure.

我们创建一个 `NeuralNetwork` 实例，把它移到 `device` 上，并打印它的结构。


In [6]:
# .to()既可以接受字符串，也可以接收 torch.device
model = NeuralNetwork().to(device)
print(model)


NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


To use the model, we pass it the input data. This executes the model\'s
`forward`, along with some [background
operations](https://github.com/pytorch/pytorch/blob/270111b7b611d174967ed204776985cefca9c144/torch/nn/modules/module.py#L866).
Do not call `model.forward()` directly!

要使用模型，我们把输入数据传给它。这会执行模型的 `forward` 方法，以及一些[后台操作](https://github.com/pytorch/pytorch/blob/270111b7b611d174967ed204776985cefca9c144/torch/nn/modules/module.py#L866)。不要直接调用 `model.forward()`！

Calling the model on the input returns a 2-dimensional tensor with dim=0
corresponding to each output of 10 raw predicted values for each class,
and dim=1 corresponding to the individual values of each output. We get
the prediction probabilities by passing it through an instance of the
`nn.Softmax` module.

对输入调用模型会返回一个二维张量：dim=0 对应每一个输出（每个输出包含各类别的 10 个原始预测值），dim=1 对应每个输出中的各个值。我们把它传给一个 `nn.Softmax` 模块实例，得到预测概率。


In [7]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([1], device='mps:0')


In [16]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
print(f"logits: {logits}")

pred_probab = nn.Softmax(dim=1)(logits)
print(f"logits after softmax: {pred_probab}")

y_pred = pred_probab.argmax(dim=1)
print(f"Predicted class: {y_pred}")


logits: tensor([[ 0.0336,  0.0419, -0.0003,  0.0434,  0.0146, -0.0583,  0.0131,  0.0537,
          0.0694, -0.0050]], device='mps:0', grad_fn=<LinearBackward0>)
logits after softmax: tensor([[0.1012, 0.1021, 0.0979, 0.1022, 0.0993, 0.0924, 0.0992, 0.1033, 0.1049,
         0.0974]], device='mps:0', grad_fn=<SoftmaxBackward0>)
Predicted class: tensor([8], device='mps:0')


------------------------------------------------------------------------


Model Layers 模型层
============

Let\'s break down the layers in the FashionMNIST model. To illustrate
it, we will take a sample minibatch of 3 images of size 28x28 and see
what happens to it as we pass it through the network.

我们来逐层拆解 FashionMNIST 模型。为了便于说明，我们取一个由 3 张 28x28 图像组成的小批量样本，看看它在通过网络时会发生什么。


In [17]:
input_image = torch.rand(3, 28, 28)
print(input_image.size())

torch.Size([3, 28, 28])


nn.Flatten
==========

We initialize the
[nn.Flatten](https://docs.pytorch.org/docs/stable/generated/torch.nn.modules.flatten.Flatten.html)
layer to convert each 2D 28x28 image into a contiguous array of 784
pixel values ( the minibatch dimension (at dim=0) is maintained).

我们初始化 [nn.Flatten](https://docs.pytorch.org/docs/stable/generated/torch.nn.modules.flatten.Flatten.html) 层，把每张 28x28 的二维图像转换为由 784 个像素值组成的连续数组（小批量维度（dim=0）保持不变）。


In [18]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


nn.Linear
=========

The [linear
layer](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)
is a module that applies a linear transformation on the input using its
stored weights and biases.

[线性层](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)是一个模块，它使用自身存储的权重和偏置对输入进行线性变换。


In [19]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


nn.ReLU
=======

Non-linear activations are what create the complex mappings between the
model\'s inputs and outputs. They are applied after linear
transformations to introduce *nonlinearity*, helping neural networks
learn a wide variety of phenomena.

非线性激活函数用于在模型的输入和输出之间建立复杂的映射。它们在线性变换之后使用，以引入*非线性*，帮助神经网络学习各种各样的现象。

In this model, we use
[nn.ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)
between our linear layers, but there\'s other activations to introduce
non-linearity in your model.

在这个模型中，我们在线性层之间使用 [nn.ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)，不过也有其他激活函数可以为模型引入非线性。


In [ ]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

In [20]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-0.0537, -0.4091, -0.3186,  0.3669,  0.0010,  0.1478,  0.2646, -0.0460,
         -0.2488, -0.1988, -0.0719,  0.0557, -0.2514,  0.1151,  0.7116, -0.5507,
         -0.6067,  0.1641, -0.4440, -0.3736],
        [ 0.0045, -0.2642, -0.5684,  0.1782,  0.0363,  0.4061,  0.7396, -0.2592,
          0.2419, -0.3459,  0.3643,  0.3205, -0.1267,  0.0110,  0.3126, -0.5845,
         -0.3486,  0.1982, -0.3115, -0.5169],
        [ 0.0785, -0.3154, -0.5770,  0.0077, -0.0234,  0.4943,  0.4365,  0.0609,
          0.1464, -0.0301,  0.4724,  0.1713, -0.2362,  0.2629,  0.4401, -0.3767,
         -0.6752,  0.1967, -0.2203, -0.3476]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0000, 0.0000, 0.0000, 0.3669, 0.0010, 0.1478, 0.2646, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0557, 0.0000, 0.1151, 0.7116, 0.0000, 0.0000, 0.1641,
         0.0000, 0.0000],
        [0.0045, 0.0000, 0.0000, 0.1782, 0.0363, 0.4061, 0.7396, 0.0000, 0.2419,
         0.0000, 0.3643, 0.3205, 0.0000, 0.0110, 0.31

nn.Sequential
=============

[nn.Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html)
is an ordered container of modules. The data is passed through all the
modules in the same order as defined. You can use sequential containers
to put together a quick network like `seq_modules`.

[nn.Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html) 是一个有序的模块容器。数据会按照定义的顺序依次通过所有模块。你可以使用顺序容器快速搭建一个网络，比如 `seq_modules`。


In [21]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3, 28, 28)
logits = seq_modules(input_image)
print(logits)

tensor([[ 0.2078, -0.1640,  0.1691, -0.2541,  0.3354,  0.0785,  0.1335,  0.3166,
         -0.1374,  0.0549],
        [ 0.2581, -0.2409,  0.1734, -0.1810,  0.3851,  0.0718,  0.0114,  0.3304,
         -0.1406, -0.0720],
        [ 0.2896, -0.0956,  0.2535, -0.2677,  0.3161,  0.1431,  0.1425,  0.2474,
         -0.1022,  0.1437]], grad_fn=<AddmmBackward0>)


nn.Softmax
==========

The last linear layer of the neural network returns [logits]{.title-ref}
- raw values in \[-infty, infty\] - which are passed to the
[nn.Softmax](https://pytorch.org/docs/stable/generated/torch.nn.Softmax.html)
module. The logits are scaled to values \[0, 1\] representing the
model\'s predicted probabilities for each class. `dim` parameter
indicates the dimension along which the values must sum to 1.

神经网络的最后一个线性层返回 *logits*，即取值范围在 \[-infty, infty\] 的原始值，然后传给 [nn.Softmax](https://pytorch.org/docs/stable/generated/torch.nn.Softmax.html) 模块。logits 会被缩放到 \[0, 1\] 之间，表示模型对每个类别的预测概率。`dim` 参数指定在哪个维度上的值之和必须为 1。


In [22]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)
print(pred_probab)

tensor([[0.1123, 0.0774, 0.1080, 0.0707, 0.1276, 0.0987, 0.1042, 0.1252, 0.0795,
         0.0964],
        [0.1193, 0.0724, 0.1096, 0.0769, 0.1355, 0.0990, 0.0932, 0.1282, 0.0801,
         0.0858],
        [0.1181, 0.0803, 0.1139, 0.0676, 0.1212, 0.1020, 0.1019, 0.1132, 0.0798,
         0.1020]], grad_fn=<SoftmaxBackward0>)


Model Parameters 模型参数
================

Many layers inside a neural network are *parameterized*, i.e. have
associated weights and biases that are optimized during training.
Subclassing `nn.Module` automatically tracks all fields defined inside
your model object, and makes all parameters accessible using your
model\'s `parameters()` or `named_parameters()` methods.

神经网络中的许多层都是*参数化*的，即带有在训练过程中被优化的权重和偏置。继承 `nn.Module` 会自动跟踪模型对象中定义的所有字段，并让你可以通过模型的 `parameters()` 或 `named_parameters()` 方法访问所有参数。

In this example, we iterate over each parameter, and print its size and
a preview of its values.

在这个例子中，我们遍历每个参数，打印它的大小和部分值的预览。


In [ ]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

In [23]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0244,  0.0247,  0.0266,  ...,  0.0168, -0.0092,  0.0343],
        [ 0.0309,  0.0221, -0.0124,  ..., -0.0040,  0.0043, -0.0192]],
       device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([-0.0080, -0.0031], device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0229, -0.0273, -0.0270,  ...,  0.0284, -0.0397, -0.0242],
        [-0.0279, -0.0198,  0.0093,  ..., -0.0298, -0.0350,  0.0064]],
       device='mps:0', grad_fn=<Slice

------------------------------------------------------------------------


Further Reading 延伸阅读
===============

-   [torch.nn API](https://pytorch.org/docs/stable/nn.html)

*   [torch.nn API 文档](https://pytorch.org/docs/stable/nn.html)
